# Orbital Validation

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / "src"))
import math
import numpy as np
from quantum_space_agi.simulation.diagnostics import specific_orbital_energy, specific_angular_momentum
from quantum_space_agi.simulation.dynamics import two_body_derivative
from quantum_space_agi.simulation.propagation import interpolate_collision_event, propagate_two_body

MU = 3.986004418e14

def strict_derivative(time_s, state, mu):
    return two_body_derivative(state, mu)

def orbital_energy(state, mu):
    return specific_orbital_energy(state, mu)

def angular_momentum_magnitude(state):
    return specific_angular_momentum(state)[1]

def linear_radius_collision_event(previous_time_s, previous_state, next_time_s, next_state, collision_radius_m):
    return interpolate_collision_event(previous_time_s, previous_state, next_time_s, next_state, collision_radius_m)

def propagate(initial, duration, dt, collision_radius=6_378_137.0):
    trajectory = propagate_two_body(initial, MU, duration, dt, collision_radius)
    return trajectory["times_s"], trajectory["states"], trajectory["collision_index"]

radius = 7_000_000.0
speed = math.sqrt(MU / radius)
initial = np.array([radius, 0, 0, 0, speed, 0], dtype=np.float64)
period = 2.0 * math.pi * math.sqrt(radius ** 3 / MU)
_, states, collision = propagate(initial, period, 20.0)
position_error = float(np.linalg.norm(states[-1, :3] - initial[:3]))
print(f"Circular period (analytic): {period}")
print(f"Elliptical period: {2.0 * math.pi * math.sqrt(8_000_000.0 ** 3 / MU)}")
print(f"Return-position error with 20 s RK4 steps: {position_error} m")
print(f"collision={collision}")
print(f"position_error={position_error}")

Circular period (analytic): 5828.516637686015
Elliptical period: 7121.081577578024
Return-position error with 20 s RK4 steps: 0.24471209409490832 m
collision=None
position_error=0.24471209409490832
